In [ ]:
import os, sys
_r = os.path.abspath(os.getcwd())
while _r != "/" and not os.path.exists(os.path.join(_r, "perm_paths.py")):
    _r = os.path.dirname(_r)
sys.path.insert(0, _r)
import perm_paths as P

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, P.FIGURES)
import figures as F

import numpy as np

# ── colours: keep identical to figure1.ipynb ───────────────────────────────
COL = {"Michaels": "#9d4edd",     # human
       "Harmange": "#e0aaff",     # human
       "Bandler":  "#48bfe3",     # mouse
       "Weinreb":  "#5aedc9"}     # mouse
SPECIES_COL = {"human": "#72369d", "mouse": "#64dfdf"}
F.DATASET_COLORS.update({F.dkey(k): v for k, v in COL.items()})

# ── THE cutoff for this notebook. Set it HERE, once. Every panel below reads
# F.TOP_N, and every file lands in panels/top<N>/ with its svg alongside.
# It is stored in the environment, so %autoreload cannot reset it mid-session.
F.use_top_n(300)          # <- the only place to change it
F.use_label_mode("celltype")

print("cutoff:", F.TOP_N, "| metric:", F.METRIC, "| methods:", list(F.METHOD_FILES))


In [ ]:
# === Source Data export helpers — run before the panel cells ===
import os, glob, numpy as np, pandas as pd

SD_ROOT = P.OUTPUTS + "/source_data"
SD_FIG  = "figure2"                      # <-- "figure1" in figure1.ipynb
SD_DIR  = f"{SD_ROOT}/{SD_FIG}"
os.makedirs(SD_DIR, exist_ok=True)

def _flat(lab):
    return str(lab).replace("\n", " ").strip()

def sd_violin(res, panel, stats=None, mem="memory", bg="background", x="group",
              min_n=None):
    """Every PLOTTED point of a violin panel, one row per value."""
    rows = [pd.DataFrame({x: _flat(lab), "set": gname,
                          "value": np.asarray(v[key], float).ravel()})
            for lab, v in res.items()
            for key, gname in (("scg_values", mem), ("non_scg_values", bg))]
    df = pd.concat(rows, ignore_index=True)
    df.to_csv(f"{SD_DIR}/{panel}_values.csv", index=False)
    print(f"  source data -> {panel}_values.csv  ({len(df):,} rows)")
    if stats is not None:
        st = pd.DataFrame(stats).copy()
        for c in st.columns:
            if st[c].dtype == object:
                st[c] = st[c].map(_flat)
        if min_n is not None and "n_mem" in st.columns:
            # the figure hides statistics for underpowered groups; say so here too
            st["tested_in_figure"] = st["n_mem"] >= min_n
        st.to_csv(f"{SD_DIR}/{panel}_stats.csv", index=False)
        print(f"  source data -> {panel}_stats.csv")
    return df

def sd_table(obj, panel, index=True):
    """A panel whose plotted content is already tabular."""
    df = pd.DataFrame(obj).copy()
    if index:
        df.index = [_flat(i) for i in df.index]
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(_flat)
    df.to_csv(f"{SD_DIR}/{panel}_values.csv", index=index)
    print(f"  source data -> {panel}_values.csv  ({len(df):,} rows)")
    return df



In [ ]:
# === Fig 2a — species / tissue / perturb-seq screen ===

# The lineage-dataset ring is GONE: it repeated figure 1a. Nothing is lost — the
# tissue names are 1:1 with the datasets and the wedge colours are figure 1a's
# palette, so each screen still says which dataset its memory genes came from.
ORDER  = ["T-lineage", "Harmange", "Bandler", "LARRY"]     # same order as Fig 1a
TISSUE = {"Michaels": "T cells", "Harmange": "Melanoma", "Bandler": "Brain", "Weinreb": "HSPC"}
SPEC   = {"Michaels": "human", "Harmange": "human", "Bandler": "mouse", "Weinreb": "mouse"}
BURST  = .30        # keep equal to BURST in figure1.ipynb

# screens per lineage dataset, from figures.py
screens = {}
for sp, ts, ds, sc in F.PERTURB_DESIGN_ROWS:
    screens.setdefault(ds, []).append(sc)

# every dataset takes 1/4 of the circle, split equally between its screens
rows, w = [], []
for k in ORDER:
    d = F.alabel(k)
    for sc in screens[d]:
        rows.append((SPEC[d], TISSUE[d], sc))     # CHANGED: dataset ring dropped
        w.append(1 / len(screens[d]))

# human|mouse boundary vertical, humans on the right (as Fig 1a)
cum   = np.concatenate([[0], np.cumsum(w)]) / sum(w) * 360
start = cum[next(i for i, r in enumerate(rows) if r[0] == "mouse")] - 90

# burst length = perturbations in that screen, keyed by the full row
PERT_N = F.perturb_screen_counts()               # measured from the data, 20-cell threshold
PERT = {r: PERT_N[r[2]] for r in rows}           # screen is index 2

fig_p, ax_p, nodes_p = F.fig_design_sunburst(
    rows=rows, weights=w, start_angle=start,
    colors=[F.DATASET_COLORS[k] for k in ORDER],
    level_colors={0: SPECIES_COL}, family_level=1,   # CHANGED: tissue carries the family
    ring_labels=None, label_gap_deg=0,
    label_orient="tangential", curved_labels=True, min_span_inside=8,
    counts=PERT, count_level=2, hub_bars=BURST > 0, count_bars="extend",  # CHANGED: leaf is level 2
    hub_scale="log", ray_span=BURST, ray_min_frac=.2,   # LOG scale -> say so in the legend
    edge_labels=None, count_at_bar=False, count_in_label=False,   # bursts, but no numbers
    fs=7, r0=.26, ring_w=.36, gap=.012, lim=1.34 + BURST + .04,   # CHANGED: .27 -> .36
    lighten_step=.12, edge_lw=1.2,
    figsize=(2.6, 2.6),                              # same size as Fig 1a: place at 100%
    save=f"{F.OUTDIR}/fig2a_design_sunburst.png", dpi=400,
)
sd_table(pd.DataFrame({"species":       [r[0] for r in rows],
                       "tissue":        [r[1] for r in rows],
                       "screen":        [r[2] for r in rows],
                       "wedge_weight":  w,
                       "perturbations": [PERT[r] for r in rows]}),
         "Fig2a", index=False)


In [ ]:
# === Fig 2c — are memory genes stronger DRIVERS? memory-gene vs non-memory knockouts ===

sys.path.insert(0, P.LIB)
import ko_axis_memory as K
import matplotlib.pyplot as plt

# Shi's memory genes from BANDLER, not Mold brain (excluded) -- same pairing as Fig 2g/h
K.SCREENS["mold"].update(memdir=f"{K.BENCH}/BandlerBrain", suffix="E13_E14",
                         memory_source="Bandler")
# Weinstock: bulk, DONOR-CORRECTED LFC (saved from weinstock_analysis.ipynb), Michaels' genes
K.SCREENS["weinstock"] = dict(
    label="Weinstock (T cells)", short="Weinstock",
    lfc=P.DATA + "/weinstock_lfc_cpm_donor_corrected.csv",
    expr=P.DATA + "/weinstock_gene_ctrl_expr.csv",
    npz=None, npz_keys=None,
    memdir=f"{K.BENCH}/T-lineage", suffix="T1_T2", memory_source="Michaels")
# Bowness: within-replicate LFC, Weinreb's genes
K.SCREENS["bowness"] = dict(
    label="Bowness (HSPC)", short="Bowness",
    lfc=P.DATA + "/velten_hspc/velten_lfc_cpm.csv",
    expr=P.DATA + "/velten_hspc/velten_gene_ctrl_expr.csv",
    npz=None, npz_keys=None,
    memdir=f"{K.BENCH}/LARRY", suffix="2.0_6.0_4.0", memory_source="Weinreb")

ORDER_B = ("marson", "weinstock", "xu", "frangieh", "mold", "bowness")   # sunburst order
LOADED = {k: K.load_screen(k, metric="mean_abs_lfc") for k in ORDER_B}

from collections import Counter

TOP_N = F.TOP_N         # GENOME-WIDE top N per method
TIER  = 2           # memory gene = called by >= TIER of the 4 methods within their top N
MIN_KO = 10         # fewer memory-gene knockouts than this: show the screen, do not test it


res_b, nmem_b, fullbg_b = {}, {}, {}
SHORTS_B = [K.SCREENS[s]["short"] for s in ORDER_B] 
for screen in ORDER_B:
    cfg = K.SCREENS[screen]
    eff, M = LOADED[screen]
    sets_gw, called_gw = K.memory_sets(cfg["memdir"], K.method_files(cfg["suffix"]),
                                       TOP_N, None)
    n_calls = Counter(g for s in sets_gw.values() for g in s)
    mem = {g for g, c in n_calls.items() if c >= TIER}
    r = K.ko_axis_results(
        screen, top_n=TOP_N, eff=eff, M=M, metric="mean_abs_lfc",
        readout="all", control="random", n_iter=1,   # the draws are discarded below
        gene_sets={f">={TIER}": mem}, exclude=called_gw,
        min_memory=1,                   # keep every screen; weak ones are marked below
        verbose=False)
    if r:
        v = list(r.values())[0]
        lab = F.slabel(cfg["short"], among=SHORTS_B, fmt="{ct}\n({author})")
        res_b[lab] = v
        nmem_b[lab] = len(np.asarray(v["scg_values"]))
        fullbg_b[lab] = F.ko_background_values(eff, M, set(mem) | set(called_gw))
print("memory-gene knockouts per screen:", nmem_b)

# ONE background rule for the whole panel: every knockout no method called, in full.
# The equal-sized random draw it replaces estimated the same pool (medians agreed to
# 0.001 on every screen with n > 1), but at a resolution that collapsed to 1 and 5
# points on the two underpowered screens.
for k, v in res_b.items():
    v["non_scg_values"] = fullbg_b[k]

fig_b, ax_b, stats_b = F.fig_responsiveness_violin(
    res=res_b, mem_label="Memory-gene knockouts", bg_label="Non-memory knockouts",
    annot="pct+stars", pct_where="left",
    show_medians=False, fill=False, median_inside=True, median_fmt="",
    xtick_rotation=35, xlabel="perturb-seq dataset",
    ylabel="mean absolute log₂ fold change\nper knockout (across all genes)",
    legend=False, fs=7, figsize=(4.8, 2.6), show=False)

# too few memory-gene knockouts to test: grey half stays, the memory violin and its
# statistics go, the handful of knockouts become points, the count replaces the stars
F.mark_underpowered(ax_b, res_b, nmem_b, min_n=MIN_KO,
                    label="too few\n(n = {n})")

F.save_fig(fig_b, f"{F.OUTDIR}/fig2c_knockouts.png", "fig2c_knockouts", dpi=400)
plt.show()
sd_violin(res_b, "Fig2c", stats=stats_b, min_n=MIN_KO,
          mem="memory-gene knockouts", bg="non-memory knockouts", x="screen")
stats_b



In [ ]:
# === Fig 2 e+f — memory genes vs control: per method (left), per matching (right) ===
import matplotlib.pyplot as plt

# left: each method's own memory-gene set vs an expression-matched background
res2a = F.responsiveness_results(top_n=F.TOP_N, timepoint="merge",
                                 n_bins=50, n_iter=100, floor=F.EXPR_FLOOR)

# right: genes called by >=2 methods, background matched on each gene property, each in
# turn and all four jointly. Detection is collinear with expression (Spearman 0.97 in
# Zhu), which makes the joint match conservative: it also matches the expression-
# detection residual (cell-type specificity), so it can only shrink a true difference.
PANELS = {
    "Expression":        ["expr"],
    "Detection":         ["detection_rate"],
    "State specificity": ["state_restriction"],
    "Gene length":       ["log_len"],
    "All four":          ["expr", "detection_rate", "state_restriction", "log_len"],
}

rung = F.matching_rung_results(
    min_methods=2, timepoint="merge", top_n=F.TOP_N,
    panels=PANELS,
    n_iter=2000, k=10, seed=0, floor=F.EXPR_FLOOR,
    background="strict", equalise=True,
    expr_bins=True, n_bins=50, expr_n_iter=100,
)

V = dict(annot="pct+stars", show_medians=False, fill=False, median_inside=True,
         pct_where="left", ylim=(0, 0.6), legend=False, fs=7, show=False)


def _unbold_numbers(ax):
    """Median values on the median lines are hardcoded bold in figures.py; set them
    to normal weight. Only purely numeric labels are touched (stars and % stay)."""
    for t in ax.texts:
        try:
            float(t.get_text().rstrip(","))
        except ValueError:
            continue
        t.set_fontweight("normal")


with plt.rc_context({**F.PUB_RC, "font.size": 7}):
    # one width unit per violin, so the gaps match on both sides (4 methods | 5 rungs)
    fig_de, (ax_d, ax_e) = plt.subplots(
        1, 2, figsize=(7.4, 3.0), sharey=True,
        gridspec_kw=dict(width_ratios=(len(res2a), len(rung))))
    _, _, stats_d = F.fig_responsiveness_violin(
        res=res2a, ax=ax_d,
        # mirrors 2b's label -- there one point is a perturbation, here it is a gene
        ylabel="mean absolute log₂ fold change\nper gene (across all perturbations)", **V)
    _, _, stats_e = F.fig_responsiveness_violin(
        res=rung, ax=ax_e, bg_label="Matched background",
        xlabel="", ylabel="", xtick_rotation=35, **V)
    ax_e.tick_params(labelleft=False)        # shared y axis: numbers only on the left
    for ax in (ax_d, ax_e):
        _unbold_numbers(ax)
    fig_de.tight_layout(w_pad=1.5)
    F.save_fig(fig_de, f"{F.OUTDIR}/fig2ef_violins.png", "fig2ef_violins", dpi=400)
    plt.show()

sd_violin(res2a, "Fig2e", stats=stats_d,
          mem="memory genes", bg="expression-matched background", x="method")
sd_violin(rung,  "Fig2f", stats=stats_e,
          mem="memory genes (≥2 methods)", bg="matched background", x="matched_on")
stats_d, stats_e



In [ ]:
lfc_df = F.load_marson_lfc()

In [ ]:
# === Fig 2i — leavout perturbations ===

# memory set = genes called by >=2 of the four methods (top N per method, groups
# merged by best within-group rank) -- the same 335 genes as the other Figure 2 panels,
# because the cutoff and the expression floor match
out = F.leaveout_curve_data(
    memset=None, min_methods=2,     # None -> consensus set instead of one method
    top_n=F.TOP_N, step=1,              # every perturbation; suffix sums make this exact and instant
    floor=F.EXPR_FLOOR,             # same universe as the rest of Figure 2
    lfc_df=lfc_df,
)

fig_l, ax_l, out = F.fig_leaveout_curve(
    out=out,
    x="n", annot_ratio=True,
    annot_dy=(0, 7),                      # was (0, 14) at fs=13
    xticks=(0, 3000, 6000), yticks=(.10, .20, .25),
    xlabel="perturbations removed",
    legend=False,
    fill_alpha=0,                         # CHANGED: no shaded band between the curves
    fs=7, figsize=(2.0, 2.2),             # CHANGED: 1.85 -> 2.6
    save=f"{F.OUTDIR}/fig2i_leaveout.png", dpi=400,
)
sd_table(out, "Fig2i", index=False)


In [ ]:
# === Fig 2 g+h — cross-dataset responsiveness: consensus lines (left), per method (right) ===

DS = F.xd_datasets()          # 6 perturb-seq screen x lineage dataset pairs

# right: each method's top-N memory genes, every screen pair
S_cut = F.xd_sweep(DS, top_ns=[F.TOP_N])
# left: genes called by >=2 / >=3 / >=4 methods
S_con = F.xd_consensus_sweep(DS, tiers=(2, 3, 4), top_n=F.TOP_N)


import numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# one_sided=True keeps only linspace(.5, 1) of the cmap it is given. Pre-stretch
# white->black across the TOP half so the slice comes back as the full ramp:
# 0 = white, max = black.
_g = plt.get_cmap("Greys")
BW = ListedColormap(np.vstack([_g(np.zeros(256)),            # unused lower half
                               _g(np.linspace(0, 1, 256))]))

ROWLAB = {d["name"]: F.slabel(F._xd_pair_parts(d["name"])[1], among=SHORTS_B,
                              fmt="{ct}\n({author})") for d in DS}

fig_gh, (ax_g, ax_h), M_gh = F.fig_cross_dataset_combo(
    S_con, S_cut, tiers=(2, 3, 4), metric="pct", cmap=BW,
    row_labels=ROWLAB,
    panel_order="heat",
    # the lines' y axis now sits on their LEFT, i.e. right where the row labels
    # were: the gap between the label column and the lines plot carries it
    gap_lines_in=.50,
    label_box=False, label_chip=False, label_air_pt=4,   # names only, no colour key
    line_color="0.82", lines_y_side="left",              # grey lines, coloured markers
    annot_fmt="{:.0f}%", annot_stars=True,               # BH stars over each value
    cbar_ticks=(),
    cbar_labels=("responsive", "less\nresponsive"),
    cbar_zero_label="0: no difference",
    cbar_below_label=None,
    cbar_zero_linespacing=1.15,
    cbar_text_rotation=90, cbar_rotate="all",
    summary_values=True,
    summary_value_pos=("above", "below", "above"),
    fs=7, figsize=(5.2, 3.0),
    heat_w_in=1.45, cbar_room_in=.62,
    xlabel_heat="", xlabel_lines="Methods calling the gene",
    save=f"{F.OUTDIR}/fig2gh_crossdataset.png", dpi=400,
)
sd_table(M_gh, "Fig2gh")
sd_table(S_con, "Fig2g_consensus_sweep", index=False)
sd_table(S_cut, "Fig2h_per_method_sweep", index=False)
M_gh.round(0)



In [ ]:
# === Fig 2j — Power-Seek memory genes from control cells: all 15 perturb-seq datasets ===
res_ps, q_ps, table_ps = F.powerseek_responsiveness()

fig_ps, ax_ps, T_ps = F.fig_powerseek_compact(
    table=table_ps,
    median=False, summary=False,        # just the points
    dot_color="0.1",                    # black
    jitter=.30, xlim=(-.55, .55), ms=24,
    #examples=True,                      # leader lines to one filled and one open point
    figsize=(2.4, 2.2), fs=7,
    name="fig2j_powerseek_compact",
    save=True, dpi=400,
)
sd_table(table_ps, "Fig2j", index=False)
table_ps[["dataset", "perturbations", "cells", "pct_higher", "q"]].round(2)


In [ ]:
# === bundle per-panel CSVs into one Source Data workbook per figure ===
import glob, os, pandas as pd

XL_MAX = 1_048_575
for fig in ("figure1", "figure2"):
    d = f"{SD_ROOT}/{fig}"
    files = sorted(glob.glob(f"{d}/*.csv"))
    if not files:
        print(f"{fig}: no CSVs yet, run its panels first"); continue
    out = f"{SD_ROOT}/Source_Data_{fig.replace('figure', 'Fig')}.xlsx"
    with pd.ExcelWriter(out, engine="openpyxl") as xw:
        for f in files:
            df = pd.read_csv(f)
            if len(df) > XL_MAX:
                print(f"  !! {os.path.basename(f)} has {len(df):,} rows, too many for one "
                      f"sheet — keeping the CSV only")
                continue
            df.to_excel(xw, sheet_name=os.path.basename(f)[:-4][:31], index=False)
    print(f"{fig} -> {out}")
